# Capstone: Build Attention from Math

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 01.01–01.10, 02.01–02.10, 03.01–03.11, 04.01–04.09, 05.01–05.07 |
| Time | ~100 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/01_build_attention_from_math.ipynb)

---

## 🎯 Learning Objective

Derive $\text{softmax}(QK^\top/\sqrt{d_k})V$ from first principles, combining linear algebra, calculus, and probability into one working attention mechanism.

---

## 📐 Theory

`01.10` introduced $\text{Attention}(Q,K,V) = \text{softmax}(QK^\top/\sqrt{d_k})V$ and implemented
it correctly, but left several pieces unmotivated: *why* $\sqrt{d_k}$ specifically, *why*
softmax rather than a simpler "pick the best match" rule, and what attention weights actually
*mean* as mathematical objects. This capstone closes every one of those gaps, using tools from
linear algebra, probability, and calculus that are each already fully built elsewhere in this
repo — the point here is synthesis, not new machinery.

### Step 1 — Why divide by $\sqrt{d_k}$: a variance calculation

For query and key vectors $\mathbf{q},\mathbf{k}\in\mathbb{R}^{d_k}$ with i.i.d., zero-mean,
unit-variance components (a reasonable approximation right after the linear projections
`01.10` builds $Q,K$ from), the dot product $\mathbf{q}\cdot\mathbf{k} = \sum_{i=1}^{d_k}
q_i k_i$ is a sum of $d_k$ independent terms. Each term $q_ik_i$ has $\mathbb{E}[q_ik_i]=0$ (by
independence and zero mean) and $\text{Var}(q_ik_i) = \mathbb{E}[q_i^2]\mathbb{E}[k_i^2] = 1$ (by
independence again — recall `03.03`'s $\text{Var}(X+Y)=\text{Var}(X)+\text{Var}(Y)$ for
independent $X,Y$). Summing $d_k$ independent, identical-variance terms gives
$\text{Var}(\mathbf{q}\cdot\mathbf{k}) = d_k$ — confirmed numerically below across several
$d_k$ values, not just claimed. Dividing by $\sqrt{d_k}$ (recall `03.03`'s
$\text{Var}(aX)=a^2\text{Var}(X)$) brings this back down to variance exactly $1$, regardless of
dimension — this is precisely `01.10`'s promised (but previously undelivered) justification.

### Step 2 — Why softmax and not hard argmax: a calculus argument

The simplest possible "attention" rule would be discrete: attend *only* to the single
best-matching key, $\text{output} = \mathbf{v}_{\arg\max_i(\text{score}_i)}$. `02.01` showed a
step-function activation gives gradient descent a zero-almost-everywhere gradient signal — this
is exactly the same failure mode: hard argmax's output is piecewise constant in the scores, so
$\partial(\text{output})/\partial(\text{score})=0$ everywhere except at score-crossing
boundaries (where it's undefined). Confirmed below: sweeping a parameter that moves the scores
continuously, hard argmax's output changes in sudden jumps with a gradient that's zero at 99%+
of points, while softmax's output changes smoothly with a gradient that's informative
everywhere. This is precisely why attention (like every trainable neural network component)
uses softmax: it's the smooth relaxation that keeps gradient-based learning possible at all.

### Step 3 — What softmax weights actually are: probability and maximum entropy

`03.02` established softmax output as a genuine categorical PMF (non-negative, sums to 1).
`05.01` went further, in passing: maximizing entropy $H(p)=-\sum_i p_i\log p_i$ subject to a
fixed expected score $\mathbb{E}_p[\text{score}]$ produces an exponential-family solution
$p_i\propto e^{\lambda\cdot\text{score}_i}$ — the *same functional form* as softmax. This
notebook makes that connection precise and verifies it directly: solving the constrained
maximum-entropy problem numerically (independent of any assumed softmax formula) reproduces
softmax's output exactly. In other words, softmax isn't an arbitrary convenient function —
it's *the* probability distribution that is maximally uncertain (spreads attention as evenly as
possible) while still being consistent with the observed scores, a genuinely principled
derivation rather than a design choice.

### Step 4 — The softmax Jacobian, derived

`02.10` used the closed-form vector-Jacobian product for backpropagating through softmax
without deriving it. Here it is, from the quotient rule: for $s_i = e^{z_i}/\sum_k e^{z_k}$,
$$\frac{\partial s_i}{\partial z_j} = \begin{cases} s_i(1-s_i) & i=j \\ -s_is_j & i\ne j\end{cases}
\;=\; s_i(\delta_{ij}-s_j)$$
(differentiate the quotient $e^{z_i}/\sum_k e^{z_k}$ with respect to $z_j$: the numerator's
derivative is $e^{z_i}$ if $i=j$ else $0$, and the denominator's derivative is always $e^{z_j}$;
combining via the quotient rule and simplifying using $s_i=e^{z_i}/\sum_k e^{z_k}$ gives exactly
the formula above). In matrix form, $J=\text{diag}(\mathbf{s})-\mathbf{s}\mathbf{s}^\top$ —
verified below against finite differences, and against 02.10's vector-Jacobian-product formula.

### Step 5 — Attention weights as a distribution with entropy

Since each row of the attention matrix is a genuine categorical PMF (Step 3), it has a
well-defined entropy (`05.01`), and that entropy has a direct interpretation: a query strongly
aligned with exactly one key produces a low-entropy, "sharp" (nearly one-hot) row — the model is
confident about where to look. A query weakly aligned with everything produces a high-entropy,
"diffuse" row, close to uniform over all keys — verified concretely below with two
constructed query vectors at the extremes.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Sweeping a single parameter that continuously shifts a set of attention scores and plotting the
resulting output under hard argmax versus softmax makes the calculus argument from the Theory
section (Step 2) directly visible: one is a discontinuous staircase with zero gradient almost
everywhere, the other is smooth everywhere.

In [ ]:
rng_viz = np.random.default_rng(0)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def hard_argmax_onehot(z):
    idx = np.argmax(z, axis=-1)
    out = np.zeros_like(z)
    out[np.arange(len(z)), idx] = 1.0
    return out

def scores_fn(theta, base_scores):
    return base_scores + theta * np.array([1.0, -0.5, 0.3, -0.2, 0.8])

base_scores = rng_viz.normal(size=5)
values = rng_viz.normal(size=5)

thetas = np.linspace(-2, 2, 400)
outputs_soft = np.array([softmax(scores_fn(t, base_scores)[None, :])[0] @ values for t in thetas])
outputs_hard = np.array([hard_argmax_onehot(scores_fn(t, base_scores)[None, :])[0] @ values for t in thetas])

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(thetas, outputs_hard, color="#C44E52", lw=2, label="hard argmax (discontinuous)")
ax.plot(thetas, outputs_soft, color="#4C72B0", lw=2, label="softmax (smooth)")
ax.set_xlabel("$\\theta$ (a parameter shifting the scores continuously)")
ax.set_ylabel("attention output")
ax.set_title("Hard argmax's output jumps in sudden steps; softmax's\noutput changes smoothly -- the calculus case for softmax")
ax.legend()
plt.show()

grad_soft = np.gradient(outputs_soft, thetas)
grad_hard = np.gradient(outputs_hard, thetas)
print(f"Softmax gradient: fraction of near-zero points (|grad|<1e-3) = {np.mean(np.abs(grad_soft) < 1e-3):.3f}")
print(f"Hard argmax gradient: fraction of near-zero points (|grad|<1e-3) = {np.mean(np.abs(grad_hard) < 1e-3):.3f}")
print(f"\nHard argmax's gradient is exactly zero almost everywhere -- gradient descent would")
print(f"receive no learning signal for the vast majority of parameter values.")

## 🐍 Implementation from Scratch

We verify each of the Theory section's five claims directly — the $\sqrt{d_k}$ variance
calculation, the softmax Jacobian against finite differences, softmax as the numerically-solved
maximum-entropy distribution, and attention-weight entropy at two extremes — then assemble every
verified piece into one complete, from-scratch scaled dot-product attention implementation.

In [ ]:
rng = np.random.default_rng(0)

# --- Step 1: verify Var(q.k) = d_k, and that dividing by sqrt(d_k) restores unit variance ---
n_trials = 100000
print("Step 1 -- variance scaling:")
for d_k in [8, 64, 512]:
    q = rng.normal(size=(n_trials, d_k))
    k = rng.normal(size=(n_trials, d_k))
    dots = np.sum(q * k, axis=1)
    print(f"  d_k={d_k:4d}: Var(q.k) = {np.var(dots):7.2f} (theory: {d_k})  |  "
          f"Var(q.k / sqrt(d_k)) = {np.var(dots / np.sqrt(d_k)):.4f} (theory: 1.0)")

# --- Step 2: hard argmax vs softmax gradient (see Visual Intuition above for the plot) ---
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

# --- Step 3: softmax IS the numerically-solved maximum-entropy distribution ---
from scipy.optimize import minimize

z_maxent = rng.normal(size=5)
lambda_true = 1.7
p_softmax = softmax(lambda_true * z_maxent)
mu_target = p_softmax @ z_maxent

def neg_entropy(p_raw):
    p = np.abs(p_raw) / np.sum(np.abs(p_raw))
    return np.sum(p * np.log(p + 1e-15))

def mean_constraint(p_raw):
    p = np.abs(p_raw) / np.sum(np.abs(p_raw))
    return p @ z_maxent - mu_target

res = minimize(neg_entropy, x0=np.ones(5) / 5, constraints=[{'type': 'eq', 'fun': mean_constraint}],
                method='SLSQP', options={'maxiter': 1000, 'ftol': 1e-12})
p_maxent_solved = np.abs(res.x) / np.sum(np.abs(res.x))

print(f"\nStep 3 -- softmax as maximum entropy, given the SAME expected-score constraint:")
print(f"  softmax(lambda*z):              {np.round(p_softmax, 5)}")
print(f"  numerically-solved max-entropy:  {np.round(p_maxent_solved, 5)}")
print(f"  Match (independent derivation reproduces softmax exactly): {np.allclose(p_softmax, p_maxent_solved, atol=1e-3)}")

# --- Step 4: the softmax Jacobian, closed form vs finite differences ---
def softmax_jacobian_closed_form(s):
    return np.diag(s) - np.outer(s, s)

def softmax_jacobian_finite_diff(z, eps=1e-6):
    n = len(z)
    J = np.zeros((n, n))
    s0 = softmax(z)
    for j in range(n):
        z_pert = z.copy(); z_pert[j] += eps
        J[:, j] = (softmax(z_pert) - s0) / eps
    return J

z_jac = rng.normal(size=5)
s_jac = softmax(z_jac)
J_closed = softmax_jacobian_closed_form(s_jac)
J_numeric = softmax_jacobian_finite_diff(z_jac)
print(f"\nStep 4 -- softmax Jacobian, diag(s) - s s^T, vs finite differences:")
print(f"  Max abs difference: {np.max(np.abs(J_closed - J_numeric)):.2e}")
print(f"  Match: {np.allclose(J_closed, J_numeric, atol=1e-4)}")
print(f"  Row sums (should be exactly 0, since every row of softmax outputs sums to a constant, 1): "
      f"{np.round(J_closed.sum(axis=1), 8)}")

# --- Step 5: entropy of attention weights, sharp vs diffuse ---
def entropy(p, axis=-1):
    p_safe = np.clip(p, 1e-15, 1)
    return -np.sum(p_safe * np.log2(p_safe), axis=axis)

d_k = 16
keys = rng.normal(size=(6, d_k))
q_sharp = keys[2] * 3.0 + rng.normal(scale=0.1, size=d_k)
q_diffuse = rng.normal(scale=0.05, size=d_k)
weights_sharp = softmax((q_sharp @ keys.T) / np.sqrt(d_k))
weights_diffuse = softmax((q_diffuse @ keys.T) / np.sqrt(d_k))
print(f"\nStep 5 -- entropy of attention weights (max possible for 6 keys: {np.log2(6):.4f} bits):")
print(f"  Sharp query (aligned with one key):   weights={np.round(weights_sharp, 3)}, entropy={entropy(weights_sharp):.4f} bits")
print(f"  Diffuse query (aligned with nothing):  weights={np.round(weights_diffuse, 3)}, entropy={entropy(weights_diffuse):.4f} bits")

# --- Full assembly: complete scaled dot-product attention, from scratch, using every verified piece ---
def scaled_dot_product_attention_from_scratch(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)   # Step 1: variance-correcting scale
    weights = softmax(scores, axis=-1)  # Step 2+3: smooth, max-entropy-justified relaxation of argmax
    return weights @ V, weights

seq_len, d_k_full, d_v = 6, 8, 4
Q_full = rng.normal(size=(seq_len, d_k_full))
K_full = rng.normal(size=(seq_len, d_k_full))
V_full = rng.normal(size=(seq_len, d_v))
output, attn_weights = scaled_dot_product_attention_from_scratch(Q_full, K_full, V_full)

# Cross-check against 01.10's einsum-based implementation style, and against a manual loop
scores_manual = np.zeros((seq_len, seq_len))
for i in range(seq_len):
    for j in range(seq_len):
        scores_manual[i, j] = Q_full[i] @ K_full[j] / np.sqrt(d_k_full)
weights_manual = softmax(scores_manual, axis=-1)
output_manual = weights_manual @ V_full

print(f"\nFull attention assembly -- matches an independent manual double-loop implementation: "
      f"{np.allclose(output, output_manual)}")
print(f"Every row of the attention matrix sums to 1 (genuine probability distribution): "
      f"{np.allclose(attn_weights.sum(axis=-1), 1.0)}")

## 🤖 Why This Matters for AI

This entire notebook *is* the AI application — every derivation above assembles directly into
the mechanism running inside every modern transformer, from BERT to GPT to the model answering
this notebook's own questions. To close the loop, we apply the entropy lens from Step 5 to a
more realistic scenario: an 8-token sequence where one token's query strongly matches an earlier
token's key (like a pronoun resolving to the noun it refers to), under `01.10`'s causal masking
(each token can only attend to itself and earlier tokens, as in autoregressive generation).
Measuring each token's attention entropy reveals a real subtlety worth being honest about: raw
entropy values aren't directly comparable across positions, because causal masking gives early
tokens a much smaller *maximum possible* entropy (token 1 can only ever look at 2 tokens, so its
entropy is capped at $\log_2 2=1$ bit, while token 7 can look at up to 8). Once that's accounted
for, the pattern is exactly what the underlying similarity structure predicts: the token with a
genuinely strong match has sharply low entropy relative to its own ceiling, while others stay
closer to their (position-dependent) maximum.

In [ ]:
rng_ai = np.random.default_rng(0)

def scaled_dot_product_attention(Q, K, V, causal_mask=False):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if causal_mask:
        seq_len = Q.shape[0]
        mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)
        scores = np.where(mask, -np.inf, scores)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# An 8-token sequence where token 4's query strongly matches token 1's key -- like a pronoun
# ("it") resolving to an earlier noun -- everything else only weakly correlated.
seq_len, d_k, d_v = 8, 16, 8
K_ai = rng_ai.normal(size=(seq_len, d_k))
Q_ai = rng_ai.normal(scale=0.3, size=(seq_len, d_k))  # realistic, post-LayerNorm-like scale
Q_ai[4] = K_ai[1] * 2.5 + rng_ai.normal(scale=0.1, size=d_k)  # deliberate strong match
V_ai = rng_ai.normal(size=(seq_len, d_v))

output_causal, weights_causal = scaled_dot_product_attention(Q_ai, K_ai, V_ai, causal_mask=True)
ent_causal = entropy(weights_causal)

# Under causal masking, token i can see i+1 tokens (indices 0..i), so its MAXIMUM possible
# entropy (uniform attention over everything visible) is log2(i+1), not a fixed constant --
# raw entropy values across positions are NOT directly comparable without this correction.
max_possible_entropy = np.log2(np.arange(1, seq_len + 1))
normalized_entropy = ent_causal / np.maximum(max_possible_entropy, 1e-9)  # position 0 has ceiling 0

print("Per-token attention entropy under causal masking:")
for i in range(seq_len):
    print(f"  token {i}: entropy={ent_causal[i]:.3f} bits, ceiling=log2({i+1})={max_possible_entropy[i]:.3f} bits, "
          f"normalized={normalized_entropy[i] if i > 0 else float('nan'):.3f}")

print(f"\nToken 4's attention weights: {np.round(weights_causal[4], 3)}")
print(f"Token 4 attends sharply to token 1 (argmax index = {np.argmax(weights_causal[4])}), the token")
print(f"its query was deliberately constructed to match -- and its NORMALIZED entropy "
      f"({normalized_entropy[4]:.3f}) is far below other tokens' (~0.7-0.95), correctly flagging it")
print(f"as unusually confident/concentrated relative to what its position could support.")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.bar(range(1, seq_len), normalized_entropy[1:], color=["#C44E52" if i == 4 else "#4C72B0" for i in range(1, seq_len)])
ax.set_xlabel("token position"); ax.set_ylabel("normalized entropy (entropy / position's own ceiling)")
ax.set_title("Token 4 (red) stands out as unusually low-entropy/confident\nonce each position's own entropy ceiling is accounted for")
ax.set_ylim(0, 1.05)
plt.show()

## 🏋️ Exercises

### Warm-up
1. Using the closed-form softmax Jacobian $J=\text{diag}(\mathbf{s})-\mathbf{s}\mathbf{s}^\top$,
   compute $J$ by hand for $\mathbf{s}=(0.5, 0.5)$ (a 2-element softmax output). Verify your
   by-hand answer against `softmax_jacobian_closed_form`, and confirm both rows sum to 0.

### Practice
2. In Step 1's variance check, the components of $q$ and $k$ are drawn i.i.d. standard normal —
   an idealization. Regenerate `q` and `k` with `scale=3.0` instead of the implicit `scale=1.0`
   (still i.i.d., just not unit-variance) and confirm empirically that $\text{Var}(q\cdot k) =
   d_k \cdot \sigma^4$ for that new scale $\sigma$, not just $d_k$. What does this tell you about
   whether $\sqrt{d_k}$ scaling alone is enough to control variance if a network's learned
   projections don't happen to produce unit-variance features?

### Challenge
3. In the AI section, change `Q_ai[4]`'s construction so it matches token **6** instead of
   token 1 (`Q_ai[4] = K_ai[6] * 2.5 + ...`) and re-run under causal masking. Since token 4
   cannot see token 6 (causal masking only allows looking backward), what happens to token 4's
   attention weights and entropy? Explain the result in terms of what the `-np.inf` masking
   actually does to `softmax`'s output for a masked position, connecting back to `01.10`'s
   causal-masking exercise.

---

## 📚 Further Reading
- Vaswani et al., [\"Attention Is All You Need\"](https://arxiv.org/abs/1706.03762)
- Jaynes, [\"Information Theory and Statistical Mechanics\"](https://bayes.wustl.edu/etj/articles/theory.1.pdf) (maximum entropy)
- Bahdanau, Cho & Bengio, [\"Neural Machine Translation by Jointly Learning to Align and Translate\"](https://arxiv.org/abs/1409.0473) (attention's precursor)

---

*Next notebook: [Capstone: Derive Backprop from Scratch](02_derive_backprop_from_scratch.ipynb)*